# `cost_eval.py`

按功能分段；每段前有标题，方便查找和修改。

### Imports & constants

In [4]:
# cost_eval.py
# 论文 Section VI 的做法：用统一的 bps 成本 c 对「所有策略」扣成本，再缩放到 15% 组合波动率。
#   每日成本 = c * (1/N_t) * sum_i |w_i,t - w_i,t-1|     （论文式 34/35，w = X * sigma_tgt / sigma）
# 本文件是独立的：benchmark 的仓位直接在这里根据特征重新计算，不需要先运行 benchmark_baz。
# 只需要：processed_data/model_ready_dataset.csv、neural_config.py、
#         mlp_oos_predictions.csv、lstm_oos_predictions.csv（样本对齐用 LSTM 的）。
# 另外加入「静态仓位」诊断：每只 ETF 固定用自己的平均仓位。

import os
import numpy as np
import pandas as pd

from neural_config import DATA_PATH, FEATURES, MACD_PAIRS

TRADING_DAYS = 252
TARGET_ANNUAL_VOL = 0.15
TARGET_DAILY_VOL = TARGET_ANNUAL_VOL / np.sqrt(TRADING_DAYS)
COSTS_BPS = [0, 1, 2, 3, 5, 10]

NN_FILES = {"MLP": "mlp_oos_predictions.csv", "LSTM": "lstm_oos_predictions.csv"}
REF_FILE = "lstm_oos_predictions.csv"         # benchmark 与它共用同一批 (Date, Ticker) 样本
STATIC_FOR = ["MLP", "LSTM"]

### `finish_panel()`

In [5]:
def finish_panel(p):
    p = p.sort_values(["Ticker", "Date"]).copy()
    p["n_t"] = p.groupby("Date")["Ticker"].transform("count")
    return p

### `load_nn_panel()`

In [6]:
def load_nn_panel(path):
    return finish_panel(pd.read_csv(path, parse_dates=["Date"]))

### `phi()`

In [7]:
def phi(y):
    return y * np.exp(-(y ** 2) / 4) / 0.89

### `benchmark_signal()`

In [8]:
def benchmark_signal(df, rule):
    if rule == "long_only":
        return np.ones(len(df))
    if rule == "sgn_returns":
        return np.sign(df["norm_ret_1y"].to_numpy())
    if rule == "macd_baz":                                    # 三个时间尺度取平均
        y = sum(df[f"macd_{s}_{l}"] for s, l in MACD_PAIRS) / len(MACD_PAIRS)
        return phi(y).to_numpy()
    raise ValueError(rule)

### `load_benchmark_base()`

与 neural_data.load_model_data 相同的过滤，并与 LSTM 的 OOS 样本对齐。

In [9]:
def load_benchmark_base():
    """与 neural_data.load_model_data 相同的过滤，并与 LSTM 的 OOS 样本对齐。"""
    df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
    df = df[df["model_available"] == True].copy()
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=FEATURES + ["vol_daily", "fwd_ret_1d"])
    df = df[df["vol_daily"] > 0]

    ref = pd.read_csv(REF_FILE, parse_dates=["Date"])[["Date", "Ticker"]]
    return df.merge(ref, on=["Date", "Ticker"], how="inner")

### `daily_gross_and_turnover()`

In [10]:
def daily_gross_and_turnover(p, weight):
    gross = weight * p["fwd_ret_1d"] / p["n_t"]
    dw = weight.groupby(p["Ticker"]).diff().abs().fillna(0.0) / p["n_t"]   # 每只票第一天视为免费建仓
    return pd.DataFrame({"Date": p["Date"], "gross": gross, "turn": dw}).groupby("Date").sum()

### `rescale()`

In [11]:
def rescale(r, span=60):
    vol = r.ewm(span=span, adjust=False).std() * np.sqrt(TRADING_DAYS)
    return (r * (TARGET_ANNUAL_VOL / vol).shift(1)).dropna()

### `sharpe()`

In [12]:
def sharpe(r):
    return r.mean() / r.std() * np.sqrt(TRADING_DAYS)

### `evaluate()`

In [13]:
def evaluate(daily, costs=COSTS_BPS):
    row = {"年化换手": daily["turn"].mean() * TRADING_DAYS}
    for c in costs:
        net = daily["gross"] - (c / 1e4) * daily["turn"]
        row[f"{c}bp"] = sharpe(rescale(net))
    return row

### `main()`

In [14]:
def main():
    rows = {}

    # ---- benchmark：直接由特征算仓位
    base = load_benchmark_base()
    for name, rule in [("Long-Only", "long_only"),
                       ("Sgn(Returns)", "sgn_returns"),
                       ("MACD-Baz2015", "macd_baz")]:
        d = base.copy()
        d["signal"] = benchmark_signal(d, rule)
        d = finish_panel(d)
        w = d["signal"] * TARGET_DAILY_VOL / d["vol_daily"]
        rows[name] = evaluate(daily_gross_and_turnover(d, w))

    # ---- 神经网络
    for name, path in NN_FILES.items():
        if not os.path.exists(path):
            print(f"[跳过] 找不到 {path}")
            continue
        p = load_nn_panel(path)
        rows[name] = evaluate(daily_gross_and_turnover(p, p["weight"]))

        if name in STATIC_FOR:
            static_w = p.groupby("Ticker")["signal"].transform("mean") * TARGET_DAILY_VOL / p["vol_daily"]
            rows[f"{name}-static(诊断)"] = evaluate(daily_gross_and_turnover(p, static_w))

    table = pd.DataFrame(rows).T
    print("缩放到 15% 组合波动率后的 Sharpe（列 = 单边交易成本，bps）\n")
    print(table.round(3).to_string())
    table.to_csv("cost_sweep_sharpe.csv")
    print("\nSaved cost_sweep_sharpe.csv")
    print("\n说明：static(诊断) 用了整个测试期的平均仓位，不是可交易策略，只用来拆分「多头偏向」和「择时」。")

### Entry point

In [15]:
if __name__ == "__main__":
    main()

缩放到 15% 组合波动率后的 Sharpe（列 = 单边交易成本，bps）

                   年化换手    0bp    1bp    2bp    3bp    5bp   10bp
Long-Only         4.538  0.487  0.481  0.476  0.470  0.459  0.431
Sgn(Returns)     16.524  0.434  0.407  0.380  0.352  0.298  0.161
MACD-Baz2015      8.416  0.377  0.360  0.342  0.324  0.288  0.198
MLP              80.124  0.632  0.399  0.166 -0.067 -0.531 -1.682
MLP-static(诊断)    1.264  0.468  0.463  0.457  0.452  0.441  0.415
LSTM             36.393  0.643  0.462  0.282  0.101 -0.259 -1.156
LSTM-static(诊断)   1.003  0.518  0.513  0.508  0.503  0.492  0.466

Saved cost_sweep_sharpe.csv

说明：static(诊断) 用了整个测试期的平均仓位，不是可交易策略，只用来拆分「多头偏向」和「择时」。
